# Nationalities (RQ1, RQ2)
The 21 Arab League nationalities that need a visa, base prompt only. Effect of a nationality = its mean distance from the other nationalities on the same application.

In [ ]:
import sys
sys.path.insert(0, "../../..")

import matplotlib.pyplot as plt
import pandas as pd

from pipeline.analysis import stats

FIGURES = stats.RESULTS / "figures"
FIGURES.mkdir(exist_ok=True)

In [ ]:
df = stats.main_comparison(stats.main_run(stats.readable(stats.load_answers())))
df = df[df['variant'] == 'base']

## Effect of each nationality

In [ ]:
effects = stats.effects(df).unstack('model')
effects.sort_values(effects.columns[0])

In [ ]:
fig, axes = plt.subplots(1, len(effects.columns), figsize=(4 * len(effects.columns), 6), sharey=True, squeeze=False)
for ax, model in zip(axes[0], effects.columns):
    effects[model].sort_values().plot.barh(ax=ax, title=model)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('score points')
fig.savefig(FIGURES / 'nationality_effects.png', dpi=200, bbox_inches='tight')

## Tests per model
Gap = highest minus lowest nationality. Spread tested by shuffling the nationality labels within each application (RQ1). Wealth order: Spearman correlation of effect and wealth rank, positive = richer countries score higher (RQ2).

In [ ]:
rows = []
for model, d in df.groupby('model'):
    spread, p_spread = stats.permutation_p(d, stats.spread)
    rho, p_rho = stats.permutation_p(d, stats.wealth_rho)
    rows.append({'model': model, 'gap': stats.gap(effects[model]), 'spread': spread, 'p_spread': p_spread, 'wealth_rho': rho, 'p_rho': p_rho})
pd.DataFrame(rows).round(3)